# 03 — Verdict final (Amendement-01)

**Logique de décision révisée** — remplace l'ancienne logique ΔAIC
simple par les branches ordonnées de l'amendement §5-§6.

## Ce qui a changé
- CPL n'entre plus dans le verdict (calculé pour contexte seulement)
- Pivot = σ_φ (depuis `02_freeform_wz_reconstruction.ipynb`)
- ΔDIC remplace ΔAIC comme critère secondaire
- Flag P (pathologie φ̇² > 0.95) comme garde-fou
- Verdict par cellule (potentiel × SNe) puis verdict global (§6)

## Entrées nécessaires
- `results/sigma_phi.json` (notebook 02)
- 4 chaînes tachyon (run01-04) + 2 chaînes ΛCDM de référence (run00, run00b)

In [ ]:
import sys, os, json
REPO = os.path.expanduser("~/Desktop/souriau-falsification")
os.chdir(REPO)
sys.path.insert(0, REPO)

import numpy as np
from getdist import loadMCSamples

RUNS = {
    ('exponentiel', 'Pantheon+'): {
        'tachyon': 'cobaya_runs/tachyon_output/run01_expo_pantheon_mac/mac_expo_pantheon',
        'lcdm':    'cobaya_runs/tachyon_output/run00_lcdm_reference/lcdm_ref',
    },
    ('exponentiel', 'DES-SN5YR'): {
        'tachyon': 'cobaya_runs/tachyon_output/run02_expo_des_mac/mac_expo_des',
        'lcdm':    'cobaya_runs/tachyon_output/run00b_lcdm_reference_des/lcdm_ref_des',
    },
    ('inverse-puissance', 'Pantheon+'): {
        'tachyon': 'cobaya_runs/tachyon_output/run03_invpower_pantheon_mac/mac_invpower_pantheon',
        'lcdm':    'cobaya_runs/tachyon_output/run00_lcdm_reference/lcdm_ref',
    },
    ('inverse-puissance', 'DES-SN5YR'): {
        'tachyon': 'cobaya_runs/tachyon_output/run04_invpower_des_mac/mac_invpower_des',
        'lcdm':    'cobaya_runs/tachyon_output/run00b_lcdm_reference_des/lcdm_ref_des',
    },
}

SEUIL_SIGMA_HIGH = 3.0   # franchissement physique robuste
SEUIL_SIGMA_LOW  = 2.0   # pas de franchissement robuste
SEUIL_DIC_LOW    = 2.0   # équivalence/préférence tachyon
SEUIL_DIC_HIGH   = 5.0   # rejet forme du modèle
SEUIL_PHIDOT2    = 0.95  # flag P: champ devient poussière

print("Setup OK — 4 cellules définies")

In [ ]:
with open('results/sigma_phi.json') as f:
    sigma_phi_data = json.load(f)

print("σ_φ chargé depuis results/sigma_phi.json :")
for sne, data in sigma_phi_data.items():
    print(f"  {sne:<12} : σ_φ = {data['sigma_phi']}")

In [ ]:
def compute_dic(samp):
    """
    DIC = chi2_mean + 2*(chi2_mean - chi2_min)
    pénalise la complexité EFFECTIVE du modèle (pas juste k - número de paramètres - brut).
    """
    chi2_chain = samp.samples[:, samp.index('chi2')]
    chi2_mean  = float(np.mean(chi2_chain))
    chi2_min   = float(np.min(chi2_chain))
    dic        = chi2_mean + 2 * (chi2_mean - chi2_min)
    return dic, chi2_mean, chi2_min

resultats = {}

for (potentiel, sne), paths in RUNS.items():
    if not os.path.exists(os.path.dirname(paths['tachyon'])):
        print(f"⚠️  Run manquant : {potentiel} × {sne}")
        continue

    samp_t = loadMCSamples(paths['tachyon'], settings={'ignore_rows': 0.3})
    samp_l = loadMCSamples(paths['lcdm'],    settings={'ignore_rows': 0.3})

    dic_t, mean_t, min_t = compute_dic(samp_t)
    dic_l, mean_l, min_l = compute_dic(samp_l)
    delta_dic = dic_t - dic_l

    names_t = [p.name for p in samp_t.getParamNames().names]
    if 'phidot_max' in names_t:
        phidot_max = samp_t.mean('phidot_max')
        flag_P = bool(phidot_max**2 > SEUIL_PHIDOT2)
    else:
        phidot_max = None
        flag_P = False
        print(f"phidot_max non trouvé pour {potentiel}×{sne}")

    resultats[(potentiel, sne)] = {
        'dic_tachyon': dic_t, 'dic_lcdm': dic_l, 'delta_dic': delta_dic,
        'phidot_max': phidot_max, 'flag_P': flag_P,
        'sigma_phi': sigma_phi_data.get(sne, {}).get('sigma_phi'),
    }

    print(f"\n{potentiel} × {sne}")
    print(f"  DIC tachyon = {dic_t:.2f}   DIC ΛCDM = {dic_l:.2f}")
    print(f"  ΔDIC        = {delta_dic:+.2f}")
    print(f"  φ̇²_max      = {phidot_max**2 if phidot_max else 'N/A'}")
    print(f"  flag P      = {flag_P}")

RÉSULTATS
Tachyon EXPONENTIEL
  H0                     = 68.1852 ± 0.3864
  Omega_m                = 0.3054 ± 0.0077
  tachyon_lambda         = 1.5412 ± 0.8640
  tachyon_phi_init       = 0.7825 ± 0.4129


TypeError: 'dict' object is not callable

In [ ]:
def verdict_cellule(sigma_phi, delta_dic, flag_P):
    """
    BRANCHES ORDONNÉES
    la première qui s'applique fixe le verdict.
    """
    if sigma_phi is not None and sigma_phi >= SEUIL_SIGMA_HIGH:
        return "DISFAVORISÉE", "franchissement physique"

    if flag_P:
        return "DISFAVORISÉE", "pathologie interne (champ → poussière)"

    if sigma_phi is not None and sigma_phi < SEUIL_SIGMA_LOW:
        if delta_dic <= SEUIL_DIC_LOW:
            return "SOUTENUE", "ajustement ≥ ΛCDM, pas de franchissement"
        elif delta_dic >= SEUIL_DIC_HIGH:
            return "DISFAVORISÉE", "forme du modèle (pas la borne -1)"

    return "NON CONCLUANT", "zone intermédiaire"


print("VERDICT PAR CELLULE (potentiel × SNe)")

verdicts_cellules = {}
for (potentiel, sne), res in resultats.items():
    verdict, motif = verdict_cellule(
        res['sigma_phi'], res['delta_dic'], res['flag_P']
    )
    verdicts_cellules[(potentiel, sne)] = verdict

    print(f"\n  {potentiel} × {sne}")
    print(f"    σ_φ = {res['sigma_phi']}, ΔDIC = {res['delta_dic']:+.2f}, "
          f"flag_P = {res['flag_P']}")
    print(f"    → {verdict} ({motif})")

In [ ]:
def verdict_global(verdicts_cellules, sigma_phi_data):
    sigma_phis = {sne: d['sigma_phi'] for sne, d in sigma_phi_data.items()}

    if all(sp is not None and sp >= 3 for sp in sigma_phis.values()):
        return "DISFAVORISÉE (franchissement physique)", \
               "σ_φ ≥ 3 pour les deux catalogues SNe"

    n_high = sum(1 for sp in sigma_phis.values() if sp is not None and sp >= 3)
    if n_high == 1:
        return "NON CONCLUANT", \
               "discordance inter-SNe sur σ_φ ≥ 3 (systématique candidate)"

    if any(v == "DISFAVORISÉE" and "pathologie" in str(verdicts_cellules)
           for v in verdicts_cellules.values()):
        return "DISFAVORISÉE (pathologie interne)", \
               "au moins une cellule porte le flag P"

    if all(v == "SOUTENUE" for v in verdicts_cellules.values()):
        return "SOUTENUE", "unanimité des 4 cellules"

    if all(v == "DISFAVORISÉE" for v in verdicts_cellules.values()):
        return "DISFAVORISÉE (forme du modèle)", "unanimité des 4 cellules"

    return "NON CONCLUANT", "zone intermédiaire ou discordance inter-cellule"


verdict_final, motif_final = verdict_global(verdicts_cellules, sigma_phi_data)

print("  VERDICT GLOBAL")
print(f"\n  {verdict_final}")
print(f"  Motif : {motif_final}")
print()
print("  Rappel : 'soutenue' et 'disfavorisée (forme)' exigent")
print("  l'unanimité des 4 cellules. Toute discordance → non concluant.")
print("  C'est le comportement voulu (amendement §6, note d'asymétrie).")

In [ ]:
import pandas as pd

rows = []
for (potentiel, sne), res in resultats.items():
    rows.append({
        'Potentiel': potentiel,
        'SNe': sne,
        'σ_φ': res['sigma_phi'],
        'ΔDIC': round(res['delta_dic'], 2),
        'flag_P': res['flag_P'],
        'Verdict cellule': verdicts_cellules[(potentiel, sne)],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f"\n{'='*70}")
print(f"  VERDICT GLOBAL : {verdict_final}")
print(f"{'='*70}")

# Sauvegarder
df.to_csv('results/verdict_cellules.csv', index=False)
with open('results/verdict_global.json', 'w') as f:
    json.dump({'verdict': verdict_final, 'motif': motif_final}, f, indent=2)
print("\nSauvegardé → results/verdict_cellules.csv, results/verdict_global.json")